# Exercise 71 — Mini-project: Schema validator

## Problem

Upstream sends you a JSON Lines file. You have an expected schema describing each column's type. For every record, you need to report:
- Missing required columns
- Type mismatches
- Extra columns (warning, not error)

A schema looks like:

```python
SCHEMA = {
    "id":     int,
    "name":   str,
    "age":    int,
    "email":  str,
    "active": bool,
}
```

`None` is allowed for any field (real data has nulls; production schemas usually mark required vs optional separately — we'll keep it simple).

Report shape:
```python
{
    "line": <1-based line number>,
    "errors": [
        "missing: <col>",
        "type: <col> expected <type> got <actual>",
    ],
    "warnings": [
        "extra: <col>",
    ],
}
```

## Setup

In [ ]:
import json
from pathlib import Path

records = [
    {"id": 1, "name": "Alice", "age": 30, "email": "a@x", "active": True},          # valid
    {"id": 2, "name": "Bob",   "age": None, "email": "b@x", "active": False},        # null age — OK
    {"id": "3", "name": "Carol", "age": 25, "email": "c@x", "active": True},         # id is str, not int
    {"id": 4, "name": "Diana", "age": 41, "email": "d@x"},                           # missing 'active'
    {"id": 5, "name": "Eve", "age": 22, "email": "e@x", "active": True, "extra": 99}, # extra column
    {"id": 6, "name": "F", "age": "old", "email": "f@x", "active": True},            # bad age type
]

JSONL = "records.jsonl"
with open(JSONL, "w", encoding="utf-8") as f:
    for r in records:
        f.write(json.dumps(r) + "\n")

SCHEMA = {
    "id": int,
    "name": str,
    "age": int,
    "email": str,
    "active": bool,
}
print("setup ok")


## Task 1 — Validate a single record

Write a function that returns `(errors, warnings)` for one record dict given the schema.

```python
def validate_record(record: dict, schema: dict) -> tuple[list[str], list[str]]:
    ...
```

Rules:
- For each `col` in `schema`:
  - If `col` not in `record`: error `f"missing: {col}"`
  - Else if `record[col]` is not `None` and not `isinstance(value, schema[col])`: error `f"type: {col} expected {schema[col].__name__} got {type(value).__name__}"`
- For each `col` in `record` not in `schema`: warning `f"extra: {col}"`

**Watch out:** in Python, `isinstance(True, int)` is `True` (`bool` is a subclass of `int`). If you check `int` before `bool`, a boolean will satisfy `int`. Check the *exact* type or special-case `bool`. Easiest fix: when `schema[col]` is `int`, reject booleans explicitly:
```python
if schema_type is int and isinstance(value, bool):
    # mismatch
```

In [ ]:
def validate_record(record: dict, schema: dict) -> tuple[list[str], list[str]]:
    pass  # TODO

errs, warns = validate_record({"id": 1, "name": "A", "age": 30, "email": "a@x", "active": True}, SCHEMA)
assert errs == [] and warns == []

errs, warns = validate_record({"id": 2, "name": "B", "age": None, "email": "b@x", "active": False}, SCHEMA)
assert errs == [] and warns == []   # None is allowed

errs, _ = validate_record({"id": "3", "name": "C", "age": 25, "email": "c@x", "active": True}, SCHEMA)
assert "type: id expected int got str" in errs

errs, _ = validate_record({"id": 4, "name": "D", "age": 41, "email": "d@x"}, SCHEMA)
assert "missing: active" in errs

errs, warns = validate_record(
    {"id": 5, "name": "E", "age": 22, "email": "e@x", "active": True, "extra": 99}, SCHEMA
)
assert errs == []
assert "extra: extra" in warns

errs, _ = validate_record({"id": 6, "name": "F", "age": True, "email": "f@x", "active": True}, SCHEMA)
assert any("type: age" in e for e in errs)        # bool is not int (with our extra rule)
print("ok")


## Task 2 — Validate a JSONL file

Write a function that processes a JSONL file line by line and returns a list of reports — only for records with at least one error or warning. Records with empty errors AND empty warnings are omitted.

```python
def validate_file(path: str, schema: dict) -> list[dict]:
    """Return a list of {'line': <1-based>, 'errors': [...], 'warnings': [...]}."""
```

Use 1-based line numbers (humans read line numbers starting at 1).

Expected for the fixture:
- Line 3 → error: id type mismatch
- Line 4 → error: missing active
- Line 5 → warning: extra `extra`
- Line 6 → error: age type mismatch

Lines 1 and 2 are clean and don't appear.

In [ ]:
import json

def validate_file(path: str, schema: dict) -> list[dict]:
    pass  # TODO

report = validate_file(JSONL, SCHEMA)
lines_in_report = {r["line"] for r in report}
assert lines_in_report == {3, 4, 5, 6}

by_line = {r["line"]: r for r in report}
assert any("type: id" in e for e in by_line[3]["errors"])
assert "missing: active" in by_line[4]["errors"]
assert by_line[5]["errors"] == []
assert "extra: extra" in by_line[5]["warnings"]
assert any("type: age" in e for e in by_line[6]["errors"])
print("ok")


## Task 3 — Summary statistics

Return a summary of the validation across the whole file as a dict:

```
{
    "total":            int,    # total records read
    "valid":            int,    # records with no errors AND no warnings
    "records_with_errors":   int,
    "records_with_warnings": int,
    "errors_by_type":   dict,   # "missing" / "type" / "extra" → count of OCCURRENCES (not records)
}
```

```python
def validation_summary(path: str, schema: dict) -> dict:
    ...
```

Note: `"extra: ..."` shows up as a warning, not an error — categorize accordingly. Categorize an issue by the **first word** before the colon (`"missing"`, `"type"`, `"extra"`).

In [ ]:
def validation_summary(path: str, schema: dict) -> dict:
    pass  # TODO

summary = validation_summary(JSONL, SCHEMA)
assert summary["total"] == 6
assert summary["valid"] == 2                # lines 1 and 2
assert summary["records_with_errors"] == 3  # lines 3, 4, 6
assert summary["records_with_warnings"] == 1  # line 5
assert summary["errors_by_type"]["type"] == 2     # line 3 (id) + line 6 (age)
assert summary["errors_by_type"]["missing"] == 1  # line 4
assert summary["errors_by_type"]["extra"] == 1    # line 5 (warning, but still counted)
print("ok")
